In [ ]:
!pip install -q transformers==4.46.3 peft==0.10.0 accelerate datasets evaluate rouge_score sentencepiece

import torch
import os
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from google.colab import drive
import evaluate
from datasets import load_dataset
from tqdm.auto import tqdm

drive.mount('/content/drive')

In [ ]:
MODEL_NAME = "VietAI/vit5-large"
ADAPTER_PATH = "/content/drive/MyDrive/Final-Statistical-Learning/Summarization/vit5-summarization-adapter"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Đang nạp trọng số LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
import json
from datasets import Dataset

NUM_SAMPLES = 10000
PATH_TO_TEST_JSON = "/content/drive/MyDrive/Final-Statistical-Learning/test_data.json"

with open(PATH_TO_TEST_JSON, 'r', encoding='utf-8') as f:
    test_data_list = json.load(f)

test_subset = Dataset.from_list(test_data_list)

def clean_data(batch):
    cleaned_articles = [doc.replace('_', ' ') for doc in batch["article"]]
    cleaned_abstracts = [abs.replace('_', ' ') for abs in batch["abstract"]]
    return {
        "article": cleaned_articles,
        "abstract": cleaned_abstracts
    }

test_subset = test_subset.map(clean_data, batched=True, batch_size=100)

In [ ]:
rouge_metric = evaluate.load("rouge")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

predictions = []
references = []

articles = [item["article"] for item in test_subset]
references = [item["abstract"] for item in test_subset]

BATCH_SIZE = 8

for i in tqdm(range(0, len(articles), BATCH_SIZE)):
    batch_articles = articles[i : i + BATCH_SIZE]
    input_texts = ["tóm tắt: " + art for art in batch_articles]

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        max_length=1024,
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=200,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    pred_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(pred_summaries)

print("\n📊 Đang tính toán điểm ROUGE...")
results = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=False)

print("="*50)
print("🏆 KẾT QUẢ ĐÁNH GIÁ (ROUGE SCORE)")
print("="*50)
print(f"📌 ROUGE-1 (Độ trùng lặp từng từ):     {results['rouge1'] * 100:.2f} %")
print(f"📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    {results['rouge2'] * 100:.2f} %")
print(f"📌 ROUGE-L (Độ trùng lặp chuỗi dài):   {results['rougeL'] * 100:.2f} %")
print(f"📌 ROUGE-Lsum (Điểm tổng hợp):         {results['rougeLsum'] * 100:.2f} %")
print("="*50)

  0%|          | 0/1250 [00:00<?, ?it/s]


📊 Đang tính toán điểm ROUGE...
🏆 KẾT QUẢ ĐÁNH GIÁ (ROUGE SCORE)
📌 ROUGE-1 (Độ trùng lặp từng từ):     57.71 %
📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    26.86 %
📌 ROUGE-L (Độ trùng lặp chuỗi dài):   36.82 %
📌 ROUGE-Lsum (Điểm tổng hợp):         36.83 %
